# MNIST + NotMNIST OOD - Figure 3(a), Colab hazırlığı

Yalnız bu notebook'u yükleyin. **Runtime > Change runtime type > GPU** (varsa T4)
ve **Runtime > Run all**. Drive izni istendiğinde onaylayın. Yeni model eğitimi
yoktur; yalnız önceki gerçek MNIST Colab run'ının 48 checkpoint'i kullanılacaktır.

Konfigürasyon sabit: Ensemble / +R / +AT / MC Dropout; **M=1,3,5**;
**seed=42,123,2026**; MNIST=10.000; NotMNIST small=18.724. Nat biriminde predictive
entropy/gap ve entropy-based AUROC; **36 seed satırı, 12 mean±sample SD özeti**.

Bu notebook'un gerçek Colab çalıştırması henüz yapılmadı. Yerel uyumluluk modu
yalnız veri/metrik/kaynak politikası testlerini yürütür; Drive veya Windows
checkpoint'lerini yükleyip gerçek inference yapılmış gibi raporlamaz.


In [ ]:
import os, sys, importlib.util, re
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['MPLBACKEND'] = 'Agg'
sys.dont_write_bytecode = True
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except (ImportError, ModuleNotFoundError, ValueError):
    IN_COLAB = False
VALIDATION_ONLY = not IN_COLAB or os.environ.get('OOD_COLAB_VALIDATE_ONLY') == '1'
EXECUTE_COLAB = IN_COLAB and not VALIDATION_ONLY

# Storage/source choices only; never change M/seeds/model/data/metrics config.
SOURCE_MODE = 'drive'                         # 'drive' or explicitly 'zip'
MNIST_RUN_ID = os.environ.get('MNIST_COLAB_RUN_ID', 'figure2a_final_local_48')
ZIP_PATH = ''                                 # zip mode: optional /content/<uploaded ZIP>
OOD_RUN_ID = os.environ.get('OOD_COLAB_RUN_ID', 'figure3a_from_' + MNIST_RUN_ID)
for value in [MNIST_RUN_ID, OOD_RUN_ID]:
    if not re.fullmatch(r'[A-Za-z0-9_-]+', value): raise ValueError('Invalid storage RUN_ID')
if SOURCE_MODE not in ['drive', 'zip']: raise ValueError('Select drive or zip explicitly')
print('Mode:', 'REAL COLAB INFERENCE' if EXECUTE_COLAB else 'LOCAL COMPATIBILITY ONLY')
print('Classifier RUN_ID:', MNIST_RUN_ID, '| source:', SOURCE_MODE, '| OOD RUN_ID:', OOD_RUN_ID)


## Ortam ve GPU
Colab'ın hazır CUDA PyTorch'u korunur. NumPy/Pillow/Matplotlib/sklearn yalnız
eksikse kurulur. OOD'de **bütün modlarda Adam.step yasaktır**; eğitim yapılmaz.


In [ ]:
import subprocess
missing = [requirement for name, requirement in [
    ('numpy', 'numpy>=1.26,<3'), ('PIL', 'Pillow>=10,<13'),
    ('matplotlib', 'matplotlib>=3.8,<4'), ('sklearn', 'scikit-learn>=1.5,<2')
] if importlib.util.find_spec(name) is None]
if missing:
    if VALIDATION_ONLY: raise RuntimeError(f'Local audit dependencies missing: {missing}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch, numpy as np, matplotlib
def _never_train(*a, **k): raise RuntimeError('No model training is permitted in OOD')
torch.optim.Adam.step = _never_train
if EXECUTE_COLAB:
    if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before Run all')
    probe = torch.ones(1, device='cuda')
    assert probe.item() == 1
    del probe
    print('GPU:', torch.cuda.get_device_name(0), '| CUDA:', torch.version.cuda)
print('PyTorch:', torch.__version__)


## Kaynak paketi ve ayrı Drive çıktı dizini
Yeni çalışma alanı ve OOD sonuçları, önceki MNIST checkpoint klasöründen ayrıdır.
ZIP seçilirse ZIP_PATH boş bırakıldığında önceki MNIST notebook'unun varsayılan
Drive ZIP yolu kullanılır; dosya yoksa yerel modellere/başka kaynağa geçilmez.


In [ ]:
import base64, gzip, hashlib, json, tempfile
PAYLOAD = '...'
EXPECTED_BUNDLE_SHA256 = '31605f0eeeac858c2cd71f960802a9edb50948666fd37501147d81d092de7aba'
packed = base64.b64decode(PAYLOAD, validate=True)
assert hashlib.sha256(packed).hexdigest() == EXPECTED_BUNDLE_SHA256
bundle = json.loads(gzip.decompress(packed))
PROJECT = Path(tempfile.mkdtemp(prefix='mnist_ood_colab_', dir=Path('/content') if IN_COLAB else None)).resolve()
for relative, encoded in bundle['assets'].items():
    path = PROJECT / relative
    if ':' in relative or '\\' in relative or not path.resolve().is_relative_to(PROJECT):
        raise ValueError('Unsafe embedded source path')
    content = base64.b64decode(encoded, validate=True)
    assert hashlib.sha256(content).hexdigest() == bundle['sha256'][relative]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(content)
(PROJECT / 'bundle_manifest.json').write_text(json.dumps(bundle['sha256'], indent=2), encoding='utf-8')
WORK = PROJECT / 'mnist_notmnist_ood'
sys.path.insert(0, str(WORK))
if EXECUTE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSISTENT = Path('/content/drive/MyDrive/DeepEnsembles_MNIST_NotMNIST_OOD') / OOD_RUN_ID
else:
    PERSISTENT = PROJECT / 'local_validation_only'
if not ZIP_PATH:
    ZIP_PATH = str(Path('/content/drive/MyDrive/DeepEnsembles_MNIST_Figure2a') / MNIST_RUN_ID / ('mnist_classification_colab_' + MNIST_RUN_ID + '.zip'))
from colab_runtime import ColabOOD
run = ColabOOD(PROJECT, PERSISTENT, SOURCE_MODE, MNIST_RUN_ID, ZIP_PATH, EXECUTE_COLAB)
RESULT = run.output
print('Isolated project:', PROJECT)
print('Separate OOD results:', RESULT)


## Gerçek Colab modellerini doğrula
Manifest, Linux/CUDA Colab kaydı, 48 model grid'i ve bağımsız MNIST audit SHA-256
değerleri kontrol edilir; her model strict yüklenir. Eksik/yanlış kayıtta hata
verilir: Windows fallback'i, alternatif kaynak arama veya eğitim yoktur.


In [ ]:
run.connect_source()


In [ ]:
run.run_tests()


## Aynı veri ve preprocessing
Resmi MNIST test dosyaları ve hash'i sabit NotMNIST small otomatik hazırlanır.
Aynı iki bozuk PNG dışlanır; tüm 10.000 / 18.724 geçerli örnek kullanılır.
Float32 / 255, 784 giriş ve native polarite korunur; ek normalize/augment yoktur.
Veri tensor hash'leri yerel doğrulanmış sürümle karşılaştırılır.


In [ ]:
run.prepare_data(
    local_ood_cache=os.environ.get('OOD_COLAB_LOCAL_DATA') if VALIDATION_ONLY else None,
    local_mnist_cache=os.environ.get('OOD_COLAB_LOCAL_MNIST') if VALIDATION_ONLY else None,
)


## Önce gerçek smoke, ardından tam OOD inference
Smoke; bütün 48 model, p=0.1 dropout/BN, farklı RNG geçişleri ve deterministik
tekrarları doğrular. Geçerse tam inference otomatik ilerler. M=1,3,5 aynı maksimum
M=5 olasılık dizisinin prefix'leridir. Figure 2(a) tutarlılığı **bu Colab
checkpoint'lerinin raw_results.csv dosyasına karşı** kontrol edilir.


In [ ]:
run.run_inference()


In [ ]:
from IPython.display import display, Image
if EXECUTE_COLAB:
    manifest = json.loads((RESULT / 'experiment_manifest.json').read_text(encoding='utf-8'))
    assert manifest['status'] == 'complete' and manifest['result_rows'] == 36 and manifest['summary_rows'] == 12
    display(Image(filename=str(RESULT / 'figures/figure3a.png'), width=1300))
    display(Image(filename=str(RESULT / 'figures/ood_auroc.png'), width=800))
    display(Image(filename=str(RESULT / 'figures/entropy_gap.png'), width=800))
    import csv
    with (RESULT / 'results_summary.csv').open(newline='', encoding='utf-8') as stream:
        for row in csv.DictReader(stream): print(row)
    print('Inference seconds:', manifest['total_inference_seconds'])
    print('Environment:', manifest['environment'])
else:
    print('No real Colab model outputs or Figure 3(a) results are claimed locally')


## Tam sonuç ZIP'i
Kaynaklar, checkpoint kaynağı/hash kayıtları, CSV/JSON, per-image entropy ve
probability cache'leri, ana/seed/ek analiz PNG+PDF'leri ve manifest saklanır.
Checkpoint ağırlıkları tekrar kopyalanmaz; kaynak MNIST Drive/ZIP'te korunur.
İndirilen ZIP benzersiz üst klasör taşır; yerel OOD sonuçlarına yazılmaz.


In [ ]:
ARCHIVE = run.export()
if ARCHIVE is not None:
    print('Complete OOD ZIP:', ARCHIVE)
    print('SHA-256:', run.report['export']['sha256'])
    from google.colab import files
    files.download(str(ARCHIVE))
else:
    print('No real Colab result ZIP is produced in local preparation')
print('Session manifest:', PERSISTENT / 'colab_session.json')
